In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

os.makedirs("figuras", exist_ok=True)

archivo = "Datos_Complementarios_CLV.xlsx"
libro = pd.ExcelFile(archivo)
print("Hojas disponibles:", libro.sheet_names)

nombre_hoja = libro.sheet_names[0]
for hoja in libro.sheet_names:
    columnas_hoja = pd.read_excel(archivo, sheet_name=hoja, nrows=0).columns.str.strip()
    if "total_neto" in columnas_hoja:
        nombre_hoja = hoja
        break

datos = pd.read_excel(archivo, sheet_name=nombre_hoja)
datos.columns = datos.columns.str.strip()
datos_originales = datos.copy()
print("Hoja seleccionada:", nombre_hoja)
print("Tamaño de la tabla:", datos.shape)
datos.head()


## 1. Revisar y preparar los datos

Se revisan los valores nulos, se completan con la mediana o la categoría más frecuente, se limitan valores extremos con el rango intercuartílico y se aplica `log(1+x)` a predictores positivos muy sesgados. Las gráficas ayudan a entender la distribución y los valores extremos.


In [ ]:
if "total_neto" in datos.columns:
    objetivo = "total_neto"
else:
    objetivo = datos.select_dtypes(include="number").columns[-1]
print("Variable objetivo:", objetivo)

print("Nulos por columna antes de limpiar:")
print(datos.isnull().sum())

# Excluir solo columnas cuyo nombre incluye tokens de identificador.
# La b?squeda por subcadena confundir?a "unidades" con el token "id".
palabras_id = {"id", "folio", "sku", "codigo", "articulo", "cp"}
columnas_id = [
    columna for columna in datos.columns
    if set(columna.lower().split("_")) & palabras_id
]

datos = datos.dropna(subset=[objetivo]).copy()

for columna in datos.columns:
    if columna == objetivo:
        continue
    if pd.api.types.is_numeric_dtype(datos[columna]):
        if datos[columna].notna().any():
            datos[columna] = datos[columna].fillna(datos[columna].median())
    else:
        moda = datos[columna].mode()
        if not moda.empty:
            datos[columna] = datos[columna].fillna(moda[0])
        else:
            datos[columna] = datos[columna].fillna("Sin dato")

columnas_numericas = datos.select_dtypes(include="number").columns.tolist()
for columna in columnas_numericas:
    if columna in columnas_id or datos[columna].nunique() < 2:
        continue
    q1 = datos[columna].quantile(0.25)
    q3 = datos[columna].quantile(0.75)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    datos[columna] = datos[columna].clip(limite_inferior, limite_superior)

columnas_transformadas = []
for columna in columnas_numericas:
    if columna == objetivo or columna in columnas_id or datos[columna].nunique() < 2:
        continue
    if datos[columna].min() >= 0 and abs(datos[columna].skew()) > 1:
        datos[columna] = np.log1p(datos[columna])
        columnas_transformadas.append(columna)

print("Columnas transformadas con log(1+x):", columnas_transformadas)
print("Nulos después de limpiar:", int(datos.isnull().sum().sum()))
print("Registros disponibles:", len(datos))

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].hist(datos[objetivo], bins=30, color="steelblue", edgecolor="white")
ejes[0].axvline(datos[objetivo].median(), color="darkorange", linestyle="--", label="Mediana")
ejes[0].set_title("Distribución de " + objetivo)
ejes[0].set_xlabel(objetivo)
ejes[0].set_ylabel("Número de registros")
ejes[0].legend()
ejes[0].grid(axis="y", alpha=0.25)
ejes[1].boxplot(datos[objetivo].dropna(), patch_artist=True, boxprops={"facecolor": "lightblue"})
ejes[1].set_title("Resumen y valores extremos de " + objetivo)
ejes[1].set_ylabel(objetivo)
ejes[1].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig("figuras/distribucion_ventas.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# Mostrar las diez marcas más frecuentes
columna_marca = next((columna for columna in datos.columns if "marc" in columna.lower()), None)
if columna_marca is not None:
    frecuencias = datos[columna_marca].value_counts().head(10).sort_values()
    frecuencias.plot(kind="barh", color="teal", figsize=(8, 5))
    plt.title("10 marcas con más registros")
    plt.xlabel("Número de registros")
    plt.ylabel("Marca")
    plt.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.savefig("figuras/marcas_frecuentes.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()


## 2. Correlaciones

La correlación de Pearson mide la relación lineal entre variables numéricas. Valores cercanos a 1 o -1 indican una relación fuerte; los cercanos a 0 indican una relación débil. El mapa de calor usa colores y etiquetas para mostrar la dirección e intensidad.


In [ ]:
variables = [
    columna for columna in datos.select_dtypes(include="number").columns
    if columna not in columnas_id and datos[columna].nunique() > 1
]

correlaciones_originales = datos_originales[variables].corr()
print("Matriz de correlaci?n original:")
print(correlaciones_originales.round(2))

plt.figure(figsize=(8, 6))
plt.imshow(correlaciones_originales, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlaci?n de Pearson")
plt.xticks(range(len(variables)), variables, rotation=45, ha="right")
plt.yticks(range(len(variables)), variables)
plt.title("Correlaci?n original, antes del preprocesamiento")
for fila in range(len(variables)):
    for columna in range(len(variables)):
        valor = correlaciones_originales.iloc[fila, columna]
        color_texto = "white" if abs(valor) > 0.55 else "black"
        plt.text(columna, fila, f"{valor:.2f}", ha="center", va="center", color=color_texto)
plt.tight_layout()
plt.savefig("figuras/correlacion_original.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

correlaciones = datos[variables].corr()
print(correlaciones.round(2))

plt.figure(figsize=(8, 6))
plt.imshow(correlaciones, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlación")
plt.xticks(range(len(variables)), variables, rotation=45, ha="right")
plt.yticks(range(len(variables)), variables)
plt.title("Correlación entre variables numéricas")
for fila in range(len(variables)):
    for columna in range(len(variables)):
        valor = correlaciones.iloc[fila, columna]
        color_texto = "white" if abs(valor) > 0.55 else "black"
        plt.text(columna, fila, f"{valor:.2f}", ha="center", va="center", color=color_texto)
plt.tight_layout()
plt.savefig("figuras/correlacion_preprocesada.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()


## 3. Regresión lineal simple y múltiple

Se reserva una parte de los datos para probar los modelos. La regresión simple usa el predictor con mayor correlación absoluta con el objetivo. La múltiple usa hasta tres predictores. Las gráficas comparan los datos reales con las predicciones y muestran el R cuadrado.


In [ ]:
predictores = [columna for columna in variables if columna != objetivo]
if len(predictores) == 0:
    raise ValueError("No hay predictores numéricos para hacer la regresión.")

X = datos[predictores]
y = datos[objetivo]
X_entrenamiento, X_prueba, y_entrenamiento, y_prueba = train_test_split(
    X, y, test_size=0.2, random_state=42
)

correlacion_entrenamiento = X_entrenamiento.corrwith(y_entrenamiento).abs().dropna()
mejor_variable = correlacion_entrenamiento.idxmax()
print("Variable con mayor correlación:", mejor_variable)
print("Correlación:", round(X_entrenamiento[mejor_variable].corr(y_entrenamiento), 3))

modelo_simple = LinearRegression()
modelo_simple.fit(X_entrenamiento[[mejor_variable]], y_entrenamiento)
predicciones_simples = modelo_simple.predict(X_prueba[[mejor_variable]])

print("\nREGRESIÓN SIMPLE")
print("Ecuación:", objetivo, "=", round(modelo_simple.intercept_, 3), "+", round(modelo_simple.coef_[0], 3), "*", mejor_variable)
print("R cuadrado en prueba:", round(modelo_simple.score(X_prueba[[mejor_variable]], y_prueba), 3))
print("Error medio absoluto:", round(np.mean(np.abs(y_prueba - predicciones_simples)), 3))

orden = np.argsort(X_prueba[mejor_variable].to_numpy())
plt.scatter(X_prueba[mejor_variable], y_prueba, alpha=0.5, label="Datos reales")
plt.plot(X_prueba[mejor_variable].to_numpy()[orden], predicciones_simples[orden], color="red", label="Recta")
plt.xlabel(mejor_variable)
plt.ylabel(objetivo)
plt.title(f"Regresión simple: {objetivo} según {mejor_variable}\nR² en prueba = {modelo_simple.score(X_prueba[[mejor_variable]], y_prueba):.3f}")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("figuras/regresion_simple.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

mejores_variables = correlacion_entrenamiento.sort_values(ascending=False).head(3).index.tolist()
modelo_multiple = LinearRegression()
modelo_multiple.fit(X_entrenamiento[mejores_variables], y_entrenamiento)
predicciones_multiples = modelo_multiple.predict(X_prueba[mejores_variables])

print("\nREGRESIÓN MÚLTIPLE")
print("Variables utilizadas:", mejores_variables)
print("Intercepto:", round(modelo_multiple.intercept_, 3))
for nombre, coeficiente in zip(mejores_variables, modelo_multiple.coef_):
    print("Coeficiente de", nombre, ":", round(coeficiente, 3))
print("R cuadrado en prueba:", round(modelo_multiple.score(X_prueba[mejores_variables], y_prueba), 3))
print("Error medio absoluto:", round(np.mean(np.abs(y_prueba - predicciones_multiples)), 3))

indice = mejores_variables.index(mejor_variable)
print("\nCoeficiente simple de", mejor_variable, ":", round(modelo_simple.coef_[0], 3))
print("Coeficiente múltiple de", mejor_variable, ":", round(modelo_multiple.coef_[indice], 3))

plt.scatter(y_prueba, predicciones_multiples, alpha=0.5, color="teal", label="Ventas")
minimo = min(y_prueba.min(), predicciones_multiples.min())
maximo = max(y_prueba.max(), predicciones_multiples.max())
plt.plot([minimo, maximo], [minimo, maximo], "r--", label="Predicción perfecta")
plt.xlabel("Valor real de " + objetivo)
plt.ylabel("Valor predicho de " + objetivo)
plt.title(f"Modelo múltiple: valores reales vs. predichos\nR² en prueba = {modelo_multiple.score(X_prueba[mejores_variables], y_prueba):.3f}")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig("figuras/regresion_multiple.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()


## 4. Regresión logística

La clase 1 representa ventas cuyo total es igual o mayor a la mediana; la clase 0 son las demás. Se equilibran las clases duplicando ejemplos de la clase minoritaria solamente en entrenamiento. La matriz de confusión permite identificar aciertos y errores por clase.


In [ ]:
umbral = datos[objetivo].median()
y_binaria = (datos[objetivo] >= umbral).astype(int)
print("Umbral para venta alta:", round(umbral, 2))
print("Cantidad por clase antes de oversampling:")
print(y_binaria.value_counts().sort_index())

variables_logisticas = [columna for columna in predictores if columna in datos.columns]
columna_marca = next((columna for columna in datos.columns if "marc" in columna.lower()), None)
if columna_marca is not None:
    variables_logisticas.append(columna_marca)

X_logistico = pd.get_dummies(datos[variables_logisticas], drop_first=True, dtype=int)
X_train, X_test, y_train, y_test = train_test_split(
    X_logistico, y_binaria, test_size=0.2, random_state=42, stratify=y_binaria
)

# Oversampling 
conteos = y_train.value_counts()
clase_menor = conteos.idxmin()
clase_mayor = conteos.max()
filas_menores = X_train[y_train == clase_menor]
filas_repetidas = filas_menores.sample(
    n=clase_mayor - conteos.min(), replace=True, random_state=42
)
clases_antes = y_train.value_counts().sort_index()
X_train_balanceado = pd.concat([X_train, filas_repetidas])
y_train_balanceado = pd.concat([y_train, y_train.loc[filas_repetidas.index]])
clases_despues = y_train_balanceado.value_counts().sort_index()

posiciones = np.arange(2)
plt.figure(figsize=(7, 4))
plt.bar(posiciones - 0.2, [clases_antes.get(0, 0), clases_antes.get(1, 0)],
        width=0.4, label=f"Antes (0: {clases_antes.get(0, 0):,}; 1: {clases_antes.get(1, 0):,})", color="steelblue")
plt.bar(posiciones + 0.2, [clases_despues.get(0, 0), clases_despues.get(1, 0)],
        width=0.4, label=f"Después (0: {clases_despues.get(0, 0):,}; 1: {clases_despues.get(1, 0):,})", color="darkorange")
plt.xticks(posiciones, ["Venta baja (0)", "Venta alta (1)"])
plt.ylabel("N\u00famero de registros de entrenamiento")
plt.title("Efecto del oversampling en el balance de clases")
plt.legend(title="Entrenamiento", loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=1, fontsize=9)
plt.grid(axis="y", alpha=0.25)
plt.tight_layout(rect=[0, 0.24, 1, 1])
plt.savefig("figuras/oversampling.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

# Ajustar la regresión logística
modelo_logistico = LogisticRegression(max_iter=1000)
modelo_logistico.fit(X_train_balanceado, y_train_balanceado)
predicciones = modelo_logistico.predict(X_test)

# Revisar el resultado en los datos de prueba
exactitud = np.mean(predicciones == y_test)
print("Exactitud en prueba:", round(exactitud, 3))
print("\nMétricas por clase en los datos de prueba:")
reporte = classification_report(
    y_test,
    predicciones,
    labels=[0, 1],
    target_names=["Venta baja (0)", "Venta alta (1)"],
    output_dict=True,
    zero_division=0,
)
tabla_metricas = pd.DataFrame(reporte).transpose()
metricas_por_clase = tabla_metricas.loc[
    ["Venta baja (0)", "Venta alta (1)"],
    ["precision", "recall", "f1-score", "support"],
]
metricas_por_clase = metricas_por_clase.rename(
    columns={
        "precision": "Precision",
        "recall": "Recall",
        "f1-score": "F1",
        "support": "Cantidad de casos",
    }
)
print(metricas_por_clase.round(3))

# Comparar precision, recall y F1 de las dos clases
metricas_por_clase[["Precision", "Recall", "F1"]].plot(
    kind="bar", figsize=(8, 5), ylim=(0, 1), color=["steelblue", "darkorange", "seagreen"]
)
plt.title("Métricas de clasificación por clase (datos de prueba)")
plt.xlabel("Clase")
plt.ylabel("Puntuación (0 a 1)")
plt.xticks(rotation=0)
plt.legend(title="Métrica")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig("figuras/metricas_por_clase.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
print("Matriz de confusión (filas = real, columnas = predicho):")
print(pd.crosstab(y_test, predicciones, rownames=["Real"], colnames=["Predicho"]))

print("\nCoeficientes logísticos:")
for nombre, coeficiente in zip(X_logistico.columns, modelo_logistico.coef_[0]):
    print(nombre, ":", round(coeficiente, 3))

conteos_prueba = y_test.value_counts().sort_index()
plt.figure(figsize=(6, 4))
plt.bar(["Venta baja (0)", "Venta alta (1)"],
        [conteos_prueba.get(0, 0), conteos_prueba.get(1, 0)],
        color=["steelblue", "darkorange"])
plt.title("Distribución de clases en los datos de prueba")
plt.ylabel("Número de ventas")
plt.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig("figuras/distribucion_clases_prueba.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()

matriz = pd.crosstab(y_test, predicciones).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
plt.figure(figsize=(5, 4))
plt.imshow(matriz, cmap="Blues")
plt.colorbar(label="Número de ventas")
plt.xticks([0, 1], ["Predicha baja", "Predicha alta"])
plt.yticks([0, 1], ["Real baja", "Real alta"])
plt.xlabel("Clase predicha")
plt.ylabel("Clase real")
plt.title(f"Matriz de confusión (exactitud = {exactitud:.3f})")
for fila in range(2):
    for columna in range(2):
        plt.text(columna, fila, str(matriz.iloc[fila, columna]), ha="center", va="center")
plt.tight_layout()
plt.savefig("figuras/matriz_confusion.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close()
